# 04 — Captura PCM síncrona de cuatro canales

Candidato BIT/HWH construido y revisado en **4.6**; este notebook
se ejecutará en placa en su paso autorizado posterior. Los hashes fijan esa
construcción; la aceptación digital y acústica en placa sigue pendiente. Este notebook **no programa la FPGA**:
se conecta al overlay ya cargado explícitamente y verifica identidad/HWH.

PCM: `int32 (N,4)`, valores signed24 extendidos, orden mic0–3, nominal48kHz.
Normalización `sample / 2**23`. La fs procede de los divisores/registros; los
UTC del host y el tiempo de llamadas no son timestamps acústicos ni miden fs.
RAW aceptado y su notebook03 se conservan.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import uuid
import numpy as np

RUN_HARDWARE = False
RUN_RAW_CHECK = False
RUN_PATTERN = False
RUN_ACOUSTICS = False
BIT_FILE = Path('/home/xilinx/acoustic_camera_phase4_candidate/pdm_pcm_capture.bit')
EXPECTED_BIT_SHA256 = '5b7b14e34d9372302132746d9ad24475c4df9a1c66f06236ace752b439e91cf2'
EXPECTED_HWH_SHA256 = 'fc757a78ae638b88eea46fa9a75a0fa67ecdaa09c2599aec19732102bdc9824f'
EXPECTED_BUILD_ID = 'B691CC9C9F6249AD938D2063E4767C54'
EVIDENCE_ROOT = Path('/home/xilinx/acoustic_camera_phase4_evidence')

# Completar en el paso físico con la posición/sonido realmente usados.
TEST_KIND = 'silence'  # silence, tone, identity, broadband
EXPECTED_TONE_HZ = None  # 1000, 2000 o3000 en ejecuciones separadas.
IDENTITY_CHANNEL = None  # Micrófono al que se aproxima la fuente.
ACOUSTIC_REPETITIONS = 10
PHYSICAL_CONTEXT = {
    'source': None, 'distance_m': None, 'orientation': None,
    'source_duration_seconds': None, 'level_setting': None,
    'position_uncertainty_m': None, 'room_notes': None,
}
LAST_RESULT = None
LAST_ANALYSIS = None


In [ ]:
if RUN_HARDWARE:
    for digest in (EXPECTED_BIT_SHA256, EXPECTED_HWH_SHA256):
        if not isinstance(digest, str) or re.fullmatch(r'[0-9a-fA-F]{64}', digest) is None:
            raise ValueError('Faltan los hashes aprobados del candidato de4.6')
    hwh_file = BIT_FILE.with_suffix('.hwh')
    for path, digest in ((BIT_FILE, EXPECTED_BIT_SHA256), (hwh_file, EXPECTED_HWH_SHA256)):
        if hashlib.sha256(path.read_bytes()).hexdigest() != digest.lower():
            raise ValueError('Hash incorrecto: ' + str(path))
    from pynq import Overlay
    from pynq_acoustic_camera import AcousticArray
    overlay = Overlay(str(BIT_FILE), download=False)
    array = AcousticArray.from_overlay(overlay, EXPECTED_BUILD_ID, provenance={
        'bit_sha256': EXPECTED_BIT_SHA256, 'hwh_sha256': EXPECTED_HWH_SHA256,
        'bit_file': str(BIT_FILE), 'hwh_file': str(hwh_file),
    })  # Rechaza si ese overlay no está cargado; no llama download().
    print(array.clock.identity)
else:
    print('Hardware desactivado; candidato local de4.6 fijado, sin prueba de placa.')


## Evidencia por ejecución

Cada sesión usa una carpeta nueva UTC+identificador, conserva `.npy` y metadatos
antes de comprobar las muestras y guarda el informe incluso si falla. La API arma
el DMA antes de START y no devuelve bloques parciales. Al terminar intenta apagar
el reloj; una recuperación fallida exige diagnosticar/restaurar el overlay antes
de reutilizarlo. No se cambia ninguna conexión desde el notebook.


In [ ]:
def save_result(directory, stem, result):
    np.save(directory / (stem + '.npy'), result.samples)
    (directory / (stem + '.json')).write_text(json.dumps(result.metadata, indent=2, allow_nan=False) + '\n')


def run_session(label, body):
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    directory = EVIDENCE_ROOT / (label + '-' + stamp + '-' + uuid.uuid4().hex[:8])
    directory.mkdir(parents=True, exist_ok=False)
    report = {'status': 'running', 'scope': 'collection_and_enabled_checks_only', 'cases': []}
    cleanup_error = None
    try:
        array.enable_clock()
        body(directory, report)
        report['status'] = 'pass'
    except BaseException as error:
        report.update(status='fail', error=type(error).__name__ + ': ' + str(error))
        if hasattr(error, 'diagnostics'):
            report['diagnostics'] = error.diagnostics
        raise
    finally:
        try:
            array.disable_clock()
        except Exception as error:
            cleanup_error = error
            report.update(status='fail', clock_cleanup_error=str(error))
        (directory / 'session-report.json').write_text(json.dumps(report, indent=2, allow_nan=False) + '\n')
    if cleanup_error is not None:
        raise RuntimeError('No se pudo confirmar apagado del reloj; ver informe') from cleanup_error
    return directory, report


## RAW breve del montaje restaurado

Mic0→B2 y Mic1→B3 fueron restaurados después del cierre de fase3 y siguen sin
retest. En el paso de placa se hará esta captura breve: bits reservados cero y
actividad de los cuatro canales. No repite toda la aceptación RAW ni identifica
por sí sola qué micrófono físico corresponde a cada canal.


In [ ]:
if RUN_HARDWARE and RUN_RAW_CHECK:
    def raw_check(directory, report):
        result = array.capture(4096, format='raw_pdm', source='microphones')
        save_result(directory, 'raw-current-wiring', result)
        bits = ((result.samples[:, None] >> np.arange(4, dtype=np.uint32)) & 1)
        fractions = bits.mean(axis=0)
        report['ones_fractions'] = fractions.tolist()
        report['transitions'] = np.count_nonzero(np.diff(bits, axis=0), axis=0).tolist()
        assert np.all((result.samples >> 4) == 0), 'Bits RAW reservados no nulos'
        assert np.all((fractions > 0) & (fractions < 1)), 'Canal RAW sin actividad'
    print(run_session('raw-wiring-check', raw_check))


## Patrón PCM: cinco tamaños × diez repeticiones

Igualdad exacta con `16*n+c`, extensión de signo, forma y longitud. El patrón
omite el filtro y requiere CLOCK_READY, pero no FILTER_READY. Esta prueba no
sustituye las pruebas digitales del decimador de4.3/4.4 ni las acústicas.


In [ ]:
if RUN_HARDWARE and RUN_PATTERN:
    def pattern_check(directory, report):
        for n in (1, 2, 17, 4096, 65536):
            expected = 16*np.arange(n, dtype=np.int32)[:, None] + np.arange(4, dtype=np.int32)
            for repetition in range(10):
                record = {'n_frames': n, 'repetition': repetition, 'status': 'running'}
                report['cases'].append(record)
                result = array.capture(n, format='pcm_s24_in_s32', source='pattern')
                save_result(directory, f'pattern-N{n}-rep{repetition:02d}', result)
                assert result.samples.dtype == np.int32 and result.samples.shape == (n, 4)
                np.testing.assert_array_equal(result.samples, expected)
                record['status'] = 'pass'
    print(run_session('pcm-pattern', pattern_check))


## Capturas físicas: bloques, tonos, identidad y retardos

En su paso físico, completar contexto y activar una prueba por sesión:

- `silence`: diez bloques de48000 tramas, sin clips/overflow/errores.
- `tone`: fuente externa moderada,1/2/3kHz en sesiones separadas; registrar
  distancia, orientación y duración. Hann y eliminación de media, pico dentro
  de±2% y≥10dB sobre mediana de potencia vecina0.5–5kHz excluyendo±100Hz.
- `identity`: aproximar la fuente a cada micrófono por separado; guardar RMS
  y posición. La identidad se documenta manualmente, sin reordenar canales.
- `broadband`: fuente centrada/equidistante; repetir retardos de los seis pares
  y registrar incertidumbre geométrica/acústica. Correlación1–3kHz, lag entero
  ±16muestras; lag positivo significa que el segundo canal llega después.

Un tono periódico puede dar retardos ambiguos. Los niveles son dBFS, no SPL;
las muestras en los extremos no equivalen necesariamente a clips. La aceptación
física completa exige el conjunto de ensayos, no solo el informe de una sesión.


In [ ]:
if RUN_HARDWARE and RUN_ACOUSTICS:
    from pynq_acoustic_camera.pcm_analysis import analyze_pcm, estimate_pair_delays
    if TEST_KIND not in ('silence', 'tone', 'identity', 'broadband'):
        raise ValueError('TEST_KIND desconocido')
    if TEST_KIND != 'silence' and any(PHYSICAL_CONTEXT[k] is None for k in ('source', 'distance_m', 'orientation', 'source_duration_seconds')):
        raise ValueError('Completar fuente, distancia, orientación y duración reales')
    if TEST_KIND == 'tone' and EXPECTED_TONE_HZ not in (1000, 2000, 3000):
        raise ValueError('Seleccionar tono nominal1000,2000o3000Hz')
    if TEST_KIND == 'identity' and IDENTITY_CHANNEL not in range(4):
        raise ValueError('Indicar micrófono0..3 junto a la fuente')

    def acoustic_check(directory, report):
        global LAST_RESULT, LAST_ANALYSIS
        report.update(kind=TEST_KIND, physical_context=PHYSICAL_CONTEXT,
                      nominal_tone_hz=EXPECTED_TONE_HZ, identity_channel=IDENTITY_CHANNEL,
                      complete_phase_acceptance=False)
        for repetition in range(ACOUSTIC_REPETITIONS):
            result = array.capture(48000, format='pcm_s24_in_s32', source='microphones')
            stem = f'{TEST_KIND}-rep{repetition:02d}'
            save_result(directory, stem, result)
            analysis = analyze_pcm(result.samples, result.metadata['nominal_sample_rate_hz'],
                                   tone_hz=EXPECTED_TONE_HZ if TEST_KIND == 'tone' else None)
            if TEST_KIND == 'broadband':
                analysis['pair_delays'] = estimate_pair_delays(result.samples)
            (directory / (stem + '-analysis.json')).write_text(json.dumps(analysis, indent=2, allow_nan=False) + '\n')
            report['cases'].append({'repetition': repetition, 'hardware_clips': result.metadata['counters']['clips'], 'analysis_file': stem + '-analysis.json'})
            LAST_RESULT, LAST_ANALYSIS = result, analysis
            assert result.metadata['counters']['clips'] == 0, 'Captura física saturada'
            if TEST_KIND == 'tone':
                assert all(c['tone_check']['pass'] for c in analysis['channels']), 'Tono fuera del criterio de banco'
    print(run_session('pcm-' + TEST_KIND, acoustic_check))


In [ ]:
if RUN_HARDWARE and RUN_ACOUSTICS and LAST_RESULT is not None:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    from pynq_acoustic_camera.pcm_analysis import pcm_spectrum
    x = LAST_RESULT.samples.astype(np.float64) / 2**23
    fs = LAST_RESULT.metadata['nominal_sample_rate_hz']
    frequencies, amplitude = pcm_spectrum(LAST_RESULT.samples, fs)
    figure = make_subplots(rows=2, cols=1, subplot_titles=('Primeros20ms', 'FFT Hann'))
    stop = min(len(x), int(0.02*fs))
    band = frequencies <= 5000
    for c in range(4):
        figure.add_trace(go.Scatter(x=np.arange(stop)/fs, y=x[:stop,c], name=f'mic{c}'), row=1, col=1)
        figure.add_trace(go.Scatter(x=frequencies[band], y=20*np.log10(np.maximum(amplitude[band,c], 1e-12)), name=f'mic{c} FFT'), row=2, col=1)
    figure.update_xaxes(title_text='Tiempo nominal (s)', row=1, col=1)
    figure.update_xaxes(title_text='Frecuencia nominal (Hz)', row=2, col=1)
    figure.update_yaxes(title_text='Fracción de FS', row=1, col=1)
    figure.update_yaxes(title_text='Amplitud dBFS', row=2, col=1)
    figure.show()


## Cierre pendiente de fase4

Registrar pruebas digitales y acústicas, límites de instrumentación y procedencia
real del candidato. Entregar el PDF exportado del BD real de Vivado aceptado.
El reloj queda apagado tras cada sesión satisfactoria; revisar el informe si el
apagado o la recuperación fallan. No modificar conexiones sin indicación física.
